In [10]:
import pandas as pd
from sklearn.metrics import precision_score, accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

In [2]:
# Scaling data in KNN is very important because KNN is distance-based — it uses distance (like Euclidean distance) to find nearest neighbors
# Suppose if features have different ranges: Age → 0 to 100 and Income → 10,000 to 10,00,000
# then Income will dominate the distance calculation, Age becomes almost irrelevant and our model gives wrong neighbors

In [3]:
heart_data = pd.read_csv("1-heart.csv")

X = heart_data.drop("target", axis=1)
y = heart_data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# K=3
knn_classifier = KNeighborsClassifier(n_neighbors=3) # The model will look at the 3 closest data points
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

# Evaluation
print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

recall score:  0.78125
accuracy score:  0.8524590163934426
precision score:  0.9259259259259259


In [5]:
# K=5
knn_classifier = KNeighborsClassifier(n_neighbors=5) # The model will look at the 5 closest data points
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))
# Here you see that evaluation score improved 

recall score:  0.875
accuracy score:  0.9016393442622951
precision score:  0.9333333333333333


In [6]:
# K=7 (Best Score till now)
knn_classifier = KNeighborsClassifier(n_neighbors=7) # The model will look at the 7 closest data points
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

# Evaluation
print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))
# Here you see that evaluation score improved 

recall score:  0.90625
accuracy score:  0.9180327868852459
precision score:  0.9354838709677419


In [7]:
# K=9
knn_classifier = KNeighborsClassifier(n_neighbors=9) # The model will look at the 9 closest data points
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

# Evaluation
print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))
# Here you see that evaluation score has decreased 

recall score:  0.875
accuracy score:  0.9016393442622951
precision score:  0.9333333333333333


In [8]:
# So we need to try using multiple hyperparameters to find the optimal value of our evaluation
# But here we actually doing hit and trial method by putting different value of hyperparameter to find optimal value
# This is not a good way of finding the value of hyperparameter instead we use GridSearchCV

# GridSearchCV for Hyperparameter Tuning

In [9]:
#  GridSearchCV stands for: Grid Search + Cross-Validation
# It is a technique in scikit-learn that helps you:
# a) Automatically find the best hyperparameters
# b) Use cross-validation to evaluate them
# c) Choose the best model based on a scoring metric


classifier = KNeighborsClassifier() # A distance-based classification algorithm that predicts a class based on the majority label among its k nearest neighbors in feature space
param_grid = {"n_neighbors": [3, 5, 7, 9]}

classifierCV = GridSearchCV(
    classifier, # the base estimator (KNN here) that will be trained repeatedly with different hyperparameter values
    param_grid, # A dictionary defining which hyperparameters and values
    cv=5, # Performs 5-fold cross-validation — splits training data into 5 parts, trains on 4, validates on 1, and repeats 5 times for stability
    scoring="recall" # By default in classification models we have accuracy but here we are using recall as the evaluation metric 
                     # to choose the best hyperparameter setting because we want model that have high recall for heart disease prediction(not accuracy)
)

classifierCV.fit(X_train_scaled, y_train)

y_pred = classifierCV.predict(X_test_scaled)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

# results
res = pd.DataFrame(classifierCV.cv_results_)
print(res[["param_n_neighbors", "mean_test_score"]])

print(classifierCV.best_params_)

recall score:  0.90625
accuracy score:  0.9180327868852459
precision score:  0.9354838709677419
   param_n_neighbors  mean_test_score
0                  3         0.864387
1                  5         0.857550
2                  7         0.871795
3                  9         0.856980
{'n_neighbors': 7}


## Pipeline

In [12]:
# Pipeline( a way to combine multiple steps (like preprocessing + model) into one object)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('knn', KNeighborsClassifier())
])

param_grid = {"knn__n_neighbors": [3, 5, 7, 9]}

classifierCV = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="recall"
)

classifierCV.fit(X_train, y_train)

y_pred = classifierCV.predict(X_test) # There is no need of providing scaling data(X_test_scaled) because the pipeline automatically 
# applies the same scaling (StandardScaler) to X_test before making predictions.

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

print(classifierCV.best_params_)

recall score:  0.90625
accuracy score:  0.9180327868852459
precision score:  0.9354838709677419
{'knn__n_neighbors': 7}
